# Практическая работа №1. Часть 3К
## Свертка и сверточный слой

**Цель:** понять, как работает операция свертки, написать сверточный слой и пулинг на «голых» тензорах, сверить их с `torch.nn.functional` и посмотреть, какие признаки выделяют простые фильтры на настоящей картинке.

> **Правило части.** В заданиях К1–К4 внутри своих функций – только срезы тензоров, арифметика, `.sum`, `.amax` и циклы `for`. Готовые `nn.Conv2d`, `F.conv2d`, `F.pad`, `F.max_pool2d`, `unfold` внутри функций использовать нельзя. Библиотечные функции – только в задании К5 (сверка).

**Задача:** сквозной пример со слайдов – картинка $5 \times 5$, в каждой строке которой стоят числа $0, 0, 0, 9, 9$ (темная левая часть и светлая правая, между ними вертикальная граница), и ядро $3 \times 3$, в каждой строке которого $-1, 0, 1$. В конце применим свои фильтры к изображению из набора FashionMNIST.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. Напечатанное значение должно совпасть с тем, что стоит после слова «ожидается». Готовые ячейки просто выполните. Сначала прочитайте теорию ниже.

## Как устроена свертка

**1. Проблема полносвязного слоя.** В части 1С картинка $28 \times 28$ вытягивалась в строку из 784 чисел, и у каждого нейрона был свой вес для каждого пикселя. Отсюда две беды:
- **много параметров**: слой 784 → 128 – это уже $100\,480$ весов;
- **нет понятия «рядом» и «то же самое, но в другом месте»**: если сдвинуть цифру на 3 пикселя, почти все 784 входа изменятся, и для сети это совсем другая картинка. Признак «вертикальная линия» приходится учить заново для каждой позиции.

Свертка решает обе проблемы одной идеей: **один маленький набор весов проходит по всей картинке**.

**2. Операция свертки.** Возьмем маленькую матрицу весов – **ядро** (фильтр) $K$ размера $k \times k$, например $3 \times 3$. Положим ядро на левый верхний угол картинки $I$, перемножим совпавшие числа и сложим – получится одно число. Сдвинем ядро на один пиксель вправо и повторим, и так по всем позициям. Для позиции $(i, j)$:

$$S(i, j) = \sum_{m=0}^{k-1}\sum_{n=0}^{k-1} I(i + m,\ j + n)\cdot K(m, n).$$

Результат $S$ называется **картой признаков** (feature map). Каждое его число показывает, насколько кусок картинки в этом месте **похож на ядро**: это то же скалярное произведение «признаки × веса», что у нейрона, только признаки берутся из окна $k \times k$.

*Пример.* Строка окна $(0, 0, 9)$ и строка ядра $(-1, 0, 1)$ дают $0 \cdot (-1) + 0 \cdot 0 + 9 \cdot 1 = 9$. Ядро $(-1, 0, 1)$ вычитает левый пиксель из правого – отвечает на вопрос «насколько справа светлее, чем слева», то есть ищет **вертикальную границу**.

**3. Ядро решает, что искать.** Одна и та же операция с разными ядрами выделяет разные признаки:
- ядро из одинаковых чисел $\frac19$ – среднее по окну, **размытие**;
- ядро Собеля $\begin{pmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{pmatrix}$ – вертикальные границы, а его транспонированная версия – горизонтальные.

В классическом компьютерном зрении такие ядра придумывали вручную. В сверточной сети ядра – это **обучаемые веса**: сеть сама находит, какие узоры полезны для задачи.

**4. Свертка или кросс-корреляция.** В математике у свертки ядро перед наложением переворачивается: $S(i, j) = \sum_{m,n} I(i - m, j - n)K(m, n)$. Формула из п. 2 без переворота правильно называется **кросс-корреляцией**. В нейросетях почти всегда используют именно ее, но по традиции называют сверткой: ядро все равно обучается, и сеть просто выучит перевернутую версию. `F.conv2d` в PyTorch тоже считает кросс-корреляцию.

**5. Размер выхода: шаг и дополнение нулями.** При картинке $n \times n$ и ядре $k \times k$ ядро помещается в $n - k + 1$ позиций по каждой оси: картинка $5 \times 5$ и ядро $3 \times 3$ дают выход $3 \times 3$. Два параметра меняют размер:
- **дополнение нулями** (padding) $p$ – вокруг картинки добавляется рамка из $p$ нулей; при $k = 3$, $p = 1$ размер выхода равен размеру входа;
- **шаг** (stride) $s$ – ядро сдвигается не на 1, а на $s$ пикселей; при $s = 2$ выход примерно вдвое меньше.

Общая формула (деление с округлением вниз):

$$n_{out} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1.$$

**6. Каналы.** Цветная картинка – это три слоя чисел (R, G, B), то есть тензор $C_{in} \times H \times W$. Ядро сверточного слоя тоже трехмерное: $C_{in} \times k \times k$ – оно проходит сразу сквозь все входные каналы и складывает все произведения в одно число. Один такой фильтр дает одну карту признаков. Слой из $C_{out}$ фильтров дает $C_{out}$ карт – **выходных каналов**. Веса слоя хранятся тензором формы $(C_{out}, C_{in}, k, k)$, смещения – вектором длины $C_{out}$ (одно смещение на фильтр). Для пакета из $N$ картинок вход имеет форму $(N, C_{in}, H, W)$, выход – $(N, C_{out}, H_{out}, W_{out})$:

$$\text{out}[n, o, i, j] = \sum_{c=0}^{C_{in}-1}\sum_{m=0}^{k-1}\sum_{t=0}^{k-1} x[n, c,\ i s + m,\ j s + t]\cdot w[o, c, m, t] + b[o].$$

**7. Сколько параметров.** У слоя $C_{out} \cdot (C_{in} \cdot k \cdot k) + C_{out}$ параметров – и это число **не зависит от размера картинки**. Слой 1 → 8 каналов с ядром $3 \times 3$: $8 \cdot 9 + 8 = 80$ параметров. Полносвязный слой с таким же выходом ($28 \cdot 28 \cdot 8 = 6272$ чисел) имел бы $784 \cdot 6272 + 6272 = 4\,923\,520$ параметров.

**8. Три идеи, которые делают свертку удобной для картинок** (Гудфеллоу, раздел 9.2):
- **локальность** (разреженные связи): каждое число выхода зависит только от окна $k \times k$, а не от всей картинки – соседние пиксели связаны сильнее далеких;
- **разделение параметров**: одно и то же ядро используется во всех позициях – отсюда 80 параметров вместо миллионов;
- **эквивариантность к сдвигу**: если сдвинуть картинку, карта признаков сдвинется так же. Признак, выученный в одном месте, находится в любом другом.

**9. Пулинг.** Слой **пулинга** (подвыборки) сжимает карту признаков: делит ее на окна (обычно $2 \times 2$ с шагом 2) и из каждого окна оставляет одно число – максимум (**max pooling**) или среднее (**average pooling**). У пулинга нет обучаемых параметров. Максимум отвечает на вопрос «есть ли признак где-то в этом окне», поэтому небольшой сдвиг картинки почти не меняет результат – появляется **инвариантность** к малым сдвигам. Кроме того, после пулинга следующие слои работают с меньшей картой – это быстрее.

**10. Рецептивное поле.** Рецептивное поле нейрона – кусок исходной картинки, от которого зависит его значение. У первого слоя с ядром $3 \times 3$ это окно $3 \times 3$. Второй слой $3 \times 3$ смотрит на $3 \times 3$ соседних чисел первого, каждое из которых видит $3 \times 3$ пикселей, – вместе окно $5 \times 5$. После $L$ слоев с ядром $k$ и шагом 1 поле равно $1 + L(k - 1)$: три слоя $3 \times 3$ видят $7 \times 7$. Пулинг и шаг $s > 1$ ускоряют рост поля. Поэтому первые слои находят простые детали (границы, углы), а глубокие – крупные части объекта.

In [ ]:
# Нужна, только если ноутбук запускается отдельно от ПР
from pathlib import Path
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display; для интерактива закомментируйте
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('torch', torch.__version__)

def show(t, k=3):
    """Печать тензора списком с округлением – так удобно сверять с «ожидается»."""
    t = torch.as_tensor(t).detach()
    if t.dim() == 0:
        return round(t.item(), k)
    return torch.round(t.double() * 10**k).div(10**k).tolist()

# FashionMNIST заранее скачивается в папку data (интернет в аудитории не нужен):
#   from torchvision import datasets; datasets.FashionMNIST('data', train=False, download=True)
DATA_DIR = Path('data')
try:
    from torchvision import datasets
    fm = datasets.FashionMNIST(DATA_DIR, train=False, download=False)
    fm_img = fm.data[0].float() / 255            # первая картинка тестовой части, 28 x 28
    fm_name = fm.classes[int(fm.targets[0])]
    print('FashionMNIST найден:', fm_name, tuple(fm_img.shape))
except Exception:
    print('FashionMNIST не найден в папке data – используется учебная картинка (скачайте набор заранее)')
    fm_img = torch.zeros(28, 28)
    fm_img[6:22, 8:14] = 1.0                     # «голенище»
    fm_img[16:22, 8:24] = 1.0                    # «подошва»
    fm_name = 'учебная картинка'

In [ ]:
# Учебный пример со слайдов
img = torch.tensor([[0., 0., 0., 9., 9.]] * 5)         # 5 x 5: слева темно, справа светло
kernel = torch.tensor([[-1., 0., 1.]] * 3)             # 3 x 3: «справа минус слева»
img_shift = torch.tensor([[0., 0., 9., 9., 9.]] * 5)   # та же граница, сдвинутая на 1 пиксель влево
print(img)

**Задание К1. Свертка одной картинки одним ядром.**

Функция `conv2d_single(img, kernel)` получает картинку `img` размера $H \times W$ и квадратное ядро `kernel` размера $k \times k$ и возвращает карту признаков размера $(H - k + 1) \times (W - k + 1)$ – без дополнения нулями и с шагом 1 (п. 2 теории).

Алгоритм:
1. узнать $k$ и размеры выхода;
2. создать тензор нулей нужного размера (`torch.zeros`);
3. двумя вложенными циклами по $i$ и $j$ вырезать окно `img[i:i + k, j:j + k]` – срез берет строки с $i$ по $i + k - 1$ включительно, то есть ровно $k$ строк;
4. перемножить окно и ядро **поэлементно** (`*`, а не `@`) и сложить все числа (`.sum()`).

*Разбор со слайда.* В каждой строке картинки $0, 0, 0, 9, 9$. Окно в столбцах 0–2 дает в каждой строке $(0, 0, 0)$, произведение с $(-1, 0, 1)$ равно 0. Окно в столбцах 1–3: строки $(0, 0, 9)$, каждая дает 9, три строки – 27. Окно в столбцах 2–4: $(0, 9, 9) \to -0 + 0 + 9 = 9$, итого 27. Все строки картинки одинаковы, поэтому и строки выхода одинаковы: $(0, 27, 27)$.

После функции есть еще две проверки: сдвиг картинки (эквивариантность, п. 8 теории) и «настоящая» свертка с перевернутым ядром (п. 4).

In [ ]:
def conv2d_single(img, kernel):
    k = kernel.shape[0]
    H_out = ...                  # <-- сколько позиций помещается по высоте
    W_out = ...                  # <-- и по ширине
    out = torch.zeros(H_out, W_out)
    for i in range(H_out):
        for j in range(W_out):
            window = ...         # <-- срез k x k, начиная с (i, j)
            out[i, j] = ...      # <-- поэлементное произведение с ядром и сумма
    return out


print(show(conv2d_single(img, kernel)), '  ожидается [[0.0, 27.0, 27.0], [0.0, 27.0, 27.0], [0.0, 27.0, 27.0]]')
print(show(conv2d_single(img_shift, kernel)[0]), '  ожидается [27.0, 27.0, 0.0] (граница сдвинулась – и ответ сдвинулся)')
print(show(conv2d_single(img, torch.flip(kernel, dims=[1]))[0]), '  ожидается [0.0, -27.0, -27.0] (перевернутое ядро)')

**Вопрос К1.** а) Почему в первом столбце выхода ноль, хотя картинка не пустая? б) Картинку сдвинули на пиксель влево – как изменилась карта признаков? Как назвать это свойство? в) Сравните с полносвязной сетью из части 1С: сколько весов нужно, чтобы найти вертикальную границу в любом месте картинки $5 \times 5$?

*Ваш ответ:*

**Задание К2. Размер выхода и число параметров.**

- `out_size(n, k, p=0, s=1)` – размер выхода по одной оси по формуле из п. 5 теории. Деление с округлением вниз в Python – оператор `//`.
- `conv_params(c_in, c_out, k)` – число параметров сверточного слоя по формуле из п. 7: $C_{out} \cdot (C_{in} \cdot k \cdot k) + C_{out}$.

*Разбор примеров.* Картинка $28 \times 28$ и ядро $3 \times 3$ с шагом 2 без дополнения: $\lfloor (28 - 3) / 2 \rfloor + 1 = \lfloor 12{,}5 \rfloor + 1 = 13$ – последний столбец картинки не попадет ни в одно окно. С ядром $5 \times 5$ и $p = 2$ размер сохраняется: $(28 + 4 - 5) / 1 + 1 = 28$. Правило для шага 1: чтобы сохранить размер, берут $p = (k - 1) / 2$.

In [ ]:
def out_size(n, k, p=0, s=1):
    return ...   # <-- (n + 2p - k) // s + 1

def conv_params(c_in, c_out, k):
    return ...   # <-- веса всех фильтров и смещения


print(out_size(5, 3), out_size(5, 3, p=1), out_size(5, 3, p=1, s=2), '  ожидается 3 5 3')
print(out_size(28, 5, p=2), out_size(28, 3, s=2), out_size(32, 5), '  ожидается 28 13 28')
print(conv_params(1, 8, 3), conv_params(3, 16, 3), '  ожидается 80 448')

**Вопрос К2.** а) Почему число параметров сверточного слоя не зависит от размера картинки, а у полносвязного зависит? б) Какие $p$ и $s$ нужно взять для ядра $5 \times 5$, чтобы картинка $32 \times 32$ превратилась в $16 \times 16$?

*Ваш ответ:*

**Задание К3. Сверточный слой: пакет, каналы, шаг, дополнение нулями.**

Функция `conv2d(x, weight, bias, stride, padding)` делает то же, что `F.conv2d`:
- `x` – пакет картинок формы $(N, C_{in}, H, W)$;
- `weight` – фильтры формы $(C_{out}, C_{in}, k, k)$;
- `bias` – смещения длины $C_{out}$;
- результат – $(N, C_{out}, H_{out}, W_{out})$, формула – в п. 6 теории.

Алгоритм:
1. **Дополнение нулями.** Создать нули формы $(N, C_{in}, H + 2p, W + 2p)$ и записать `x` в середину: `x_pad[:, :, p:p + H, p:p + W] = x`.
2. **Размер выхода** – своей функцией `out_size`.
3. **Два цикла по позициям** $(i, j)$ выхода. Окно для всех картинок и всех входных каналов сразу: `x_pad[:, :, i*s : i*s + k, j*s : j*s + k]`, форма $(N, C_{in}, k, k)$.
4. **Все фильтры сразу через транслирование.** `window.unsqueeze(1)` имеет форму $(N, 1, C_{in}, k, k)$, `weight` – $(C_{out}, C_{in}, k, k)$. Их произведение – $(N, C_{out}, C_{in}, k, k)$: каждое окно умножено на каждый фильтр. Сумма по осям $(2, 3, 4)$ дает $(N, C_{out})$ – это `out[:, :, i, j]`.
5. **Смещения** прибавить в конце: `bias.view(1, -1, 1, 1)` имеет форму $(1, C_{out}, 1, 1)$ и прибавится к каждой карте своего канала.

*Проверка на примере.* Два фильтра: вертикальный (наше ядро) и горизонтальный (оно же, транспонированное), смещения $(0, 1)$. На нашей картинке нет горизонтальных перепадов, поэтому вторая карта – одни единицы (только смещение). С дополнением $p = 1$ у краев появляются числа 18 и $-18$: окно наполовину лежит на нулевой рамке, и на правом краю светлая картинка резко «обрывается» в ноль – дополнение нулями создает искусственные границы.

In [ ]:
def conv2d(x, weight, bias, stride=1, padding=0):
    N, C_in, H, W = x.shape
    C_out, _, k, _ = weight.shape
    p, s = padding, stride
    x_pad = torch.zeros(N, C_in, H + 2 * p, W + 2 * p)
    x_pad[:, :, p:p + H, p:p + W] = ...           # <-- исходный пакет в середину
    H_out, W_out = ..., ...                       # <-- своя out_size по высоте и ширине
    out = torch.zeros(N, C_out, H_out, W_out)
    for i in range(H_out):
        for j in range(W_out):
            window = ...                          # <-- окно (N, C_in, k, k) с учетом шага s
            out[:, :, i, j] = ...                 # <-- (window.unsqueeze(1) * weight), сумма по осям (2, 3, 4)
    return out + ...                              # <-- смещения формы (1, C_out, 1, 1)


x_ex = img.view(1, 1, 5, 5)
w_ex = torch.stack([kernel, kernel.T]).view(2, 1, 3, 3)   # вертикальный и горизонтальный фильтры
b_ex = torch.tensor([0., 1.])

out = conv2d(x_ex, w_ex, b_ex)
print(tuple(out.shape), '  ожидается (1, 2, 3, 3)')
print(show(out[0, 0, 0]), show(out[0, 1]), '  ожидается [0.0, 27.0, 27.0] и все единицы')
out_p = conv2d(x_ex, w_ex, b_ex, padding=1)
print(tuple(out_p.shape), '  ожидается (1, 2, 5, 5)')
print(show(out_p[0, 0, 0]), show(out_p[0, 0, 2]), '  ожидается [0.0, 0.0, 18.0, 18.0, -18.0] [0.0, 0.0, 27.0, 27.0, -27.0]')
out_s = conv2d(x_ex, w_ex, b_ex, stride=2, padding=1)
print(show(out_s[0, 0]), '  ожидается [[0.0, 18.0, -18.0], [0.0, 27.0, -27.0], [0.0, 18.0, -18.0]]')

**Вопрос К3.** а) Откуда в выходе с дополнением нулями взялось $-27$ в правом столбце? б) Сколько умножений делает слой 3 → 16 с ядром $3 \times 3$ на одной картинке $32 \times 32$ при $p = 1$? Сравните с числом его параметров.

*Ваш ответ:*

**Задание К4. Max pooling.**

Функция `max_pool2d(x, k, s=None)` для пакета карт $(N, C, H, W)$ возвращает $(N, C, H_{out}, W_{out})$: в каждом окне $k \times k$ (шаг $s$, по умолчанию $s = k$ – окна не перекрываются) остается максимум (п. 9 теории). Дополнения нулями нет, размер выхода – `out_size(H, k, 0, s)`.

Окно для всех картинок и каналов: `x[:, :, i*s : i*s + k, j*s : j*s + k]`. Максимум сразу по двум последним осям дает `window.amax(dim=(2, 3))` – форма $(N, C)$.

*Пример.* Карта $4 \times 4$ с числами от 0 до 15 по строкам. Окна $2 \times 2$: левое верхнее $\{0, 1, 4, 5\}$ – максимум 5; правое верхнее $\{2, 3, 6, 7\}$ – 7; нижние – 13 и 15. Выход $\begin{pmatrix}5&7\\13&15\end{pmatrix}$ – вчетверо меньше чисел.

In [ ]:
def max_pool2d(x, k, s=None):
    s = k if s is None else s
    N, C, H, W = x.shape
    H_out, W_out = ..., ...                     # <-- out_size без дополнения
    out = torch.zeros(N, C, H_out, W_out)
    for i in range(H_out):
        for j in range(W_out):
            out[:, :, i, j] = ...               # <-- максимум окна по осям (2, 3)
    return out


grid = torch.arange(16.).view(1, 1, 4, 4)
print(show(max_pool2d(grid, 2)[0, 0]), '  ожидается [[5.0, 7.0], [13.0, 15.0]]')
fmap = conv2d_single(img, kernel).view(1, 1, 3, 3)
fmap_shift = conv2d_single(img_shift, kernel).view(1, 1, 3, 3)
print(show(max_pool2d(fmap, 3)), show(max_pool2d(fmap_shift, 3)), '  ожидается [[[[27.0]]]] [[[[27.0]]]] (сдвиг не изменил ответ)')
print(show(max_pool2d(fmap, 2, s=1)[0, 0]), '  ожидается [[27.0, 27.0], [27.0, 27.0]]')

**Вопрос К4.** а) Карта признаков после сдвига картинки изменилась, а максимум по ней – нет. Чем эквивариантность отличается от инвариантности? б) Сколько обучаемых параметров у слоя max pooling? в) Какая информация теряется при пулинге?

*Ваш ответ:*

**Задание К5. Сверка с PyTorch.**

Функция `max_diff_with_torch(x, weight, bias, stride, padding)` возвращает наибольшее по модулю расхождение между своей `conv2d` и библиотечной `F.conv2d` с теми же параметрами (одно число, `.item()`). Проверяем на случайном пакете из двух трехканальных картинок $8 \times 8$ и четырех фильтрах – это «честная» проверка всех осей сразу: пакета, входных и выходных каналов.

Расхождение порядка $10^{-6}$ нормально: числа хранятся в `float32` (около 7 значащих цифр), а PyTorch складывает 27 произведений в другом порядке, из-за чего округления получаются чуть иными.

In [ ]:
def max_diff_with_torch(x, weight, bias, stride=1, padding=0):
    mine = ...        # <-- своя conv2d
    ref = ...         # <-- F.conv2d(x, weight, bias, stride=stride, padding=padding)
    return ...        # <-- наибольшее |mine - ref|, число


g = torch.Generator().manual_seed(RANDOM_STATE)
x_r = torch.randn(2, 3, 8, 8, generator=g)
w_r = torch.randn(4, 3, 3, 3, generator=g)
b_r = torch.randn(4, generator=g)
for s, p in [(1, 0), (1, 1), (2, 1)]:
    d = max_diff_with_torch(x_r, w_r, b_r, stride=s, padding=p)
    print(f'stride={s}, padding={p}: {d:.1e}', '  ожидается < 1e-05')
print(f'{(max_pool2d(x_r, 2) - F.max_pool2d(x_r, 2)).abs().max().item():.1e}', '  ожидается 0.0e+00 (максимум не округляется)')
print(sum(p.numel() for p in nn.Conv2d(1, 8, 3).parameters()), conv_params(1, 8, 3), '  ожидается 80 80')

**Вопрос К5.** а) Почему у max pooling расхождение ровно ноль, а у свертки – нет? б) `nn.Conv2d(1, 8, 3).weight` имеет форму $(8, 1, 3, 3)$. Что означает каждая ось?

*Ваш ответ:*

**Задание К6. Фильтры на настоящей картинке.**

Функция `make_filters()` возвращает тензор формы $(3, 1, 3, 3)$ – три фильтра для черно-белой картинки (п. 3 теории):
1. размытие – все девять чисел равны $\frac19$ (среднее по окну);
2. вертикальные границы (Собель) $\begin{pmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{pmatrix}$ – центральная строка весит вдвое больше, так фильтр меньше реагирует на шум;
3. горизонтальные границы – транспонированный фильтр 2 (`.T`).

Применим их своей `conv2d` с $p = 1$ к картинке из FashionMNIST (форма $(1, 1, 28, 28)$) и получим три карты признаков $28 \times 28$. Силу границы в каждой точке без учета направления дает $\sqrt{g_x^2 + g_y^2}$, где $g_x$ и $g_y$ – отклики фильтров 2 и 3.

*Проверка на учебном примере.* Собель по вертикали на нашей картинке: в каждом окне три строки с весами $1, 2, 1$, каждая дает 9, итого $9 + 18 + 9 = 36$ – выход $(0, 36, 36)$. Размытие: окно из шести девяток и трех нулей дает $54 / 9 = 6$.

In [ ]:
def make_filters():
    blur = ...                                    # <-- 3 x 3, все числа 1/9
    sobel_x = torch.tensor(...)                   # <-- вертикальные границы
    sobel_y = ...                                 # <-- горизонтальные: транспонированный sobel_x
    return torch.stack([blur, sobel_x, sobel_y]).view(3, 1, 3, 3)


filters = make_filters()
print(tuple(filters.shape), show(filters[0].sum()), '  ожидается (3, 1, 3, 3) 1.0')
maps_ex = conv2d(x_ex, filters, torch.zeros(3))
print(show(maps_ex[0, 0, 0]), show(maps_ex[0, 1, 0]), show(maps_ex[0, 2, 0]),
      '  ожидается [0.0, 3.0, 6.0] [0.0, 36.0, 36.0] [0.0, 0.0, 0.0]')

x_fm = fm_img.view(1, 1, 28, 28)
maps = conv2d(x_fm, filters, torch.zeros(3), padding=1)
edges = torch.sqrt(maps[0, 1] ** 2 + maps[0, 2] ** 2)
print(tuple(maps.shape), tuple(edges.shape), '  ожидается (1, 3, 28, 28) (28, 28)')

In [ ]:
# Готовая ячейка: исходная картинка и карты признаков
titles = ['исходная', 'размытие', 'вертикальные границы', 'горизонтальные границы', 'сила границы']
pics = [fm_img, maps[0, 0], maps[0, 1], maps[0, 2], edges]
fig, axs = plt.subplots(1, 5, figsize=(13, 3))
for ax, t, p in zip(axs, titles, pics):
    ax.imshow(p.detach(), cmap='gray'); ax.set_title(t, fontsize=10); ax.axis('off')
fig.suptitle(f'Фильтры 3 x 3 на картинке «{fm_name}»')
fig.tight_layout(); fig.savefig('feature_maps.png', dpi=110); plt.close(fig)
display(Image('feature_maps.png'))

**Вопрос К6.** а) На карте вертикальных границ одни края светлые, другие темные. Почему? б) Чем обученные ядра сверточной сети отличаются от ядер Собеля? в) Сравните с методами 9 класса: почему для kNN или дерева решений признаки картинки часто приходилось готовить вручную?

*Ваш ответ:*

### Задание повышенной сложности (к части 3К)

1. **Свертка через матричное умножение.** Функция `F.unfold` вырезает все окна сразу и укладывает их столбцами матрицы (этот прием называется im2col). Напишите `conv2d_unfold(x, weight, bias, stride, padding)` через `F.unfold` и одно умножение матриц, сверьте с `F.conv2d` и замерьте время (`time.perf_counter`) на пакете из 64 картинок $28 \times 28$ для своей `conv2d`, своей `conv2d_unfold` и `F.conv2d`. Во сколько раз ускорение?
2. **Рецептивное поле через градиенты.** Соберите три слоя `nn.Conv2d(1, 1, 3)` подряд, подайте картинку $15 \times 15$ с `requires_grad=True`, возьмите центральное число выхода и вызовите `backward()`. Нарисуйте `x.grad != 0`: какого размера область? Сравните с формулой $1 + L(k - 1)$.
3. **Эквивариантность на настоящей картинке.** Сдвиньте картинку FashionMNIST на 3 пикселя вправо (`torch.roll` по последней оси) и проверьте, что карта вертикальных границ сдвинулась так же. Где равенство нарушается и почему?
4. **Настоящая свертка.** Напишите `true_conv2d_single` с переворотом ядра (`torch.flip`) и покажите, что для симметричного ядра размытия результаты совпадают, а для ядра Собеля – отличаются знаком.

In [ ]:
# Место для задания повышенной сложности